# NB4 v2 — GraphCast SEA (Residual + 2-Timestep + Static-as-Input)

**3 thay đổi so với v1:**
1. **Residual prediction**: model dự báo `ΔX = X[t+1] − X[t]` thay vì X[t+1] trực tiếp. Output = x_dyn_t + delta.
2. **2-timestep input**: mỗi sample feed cả `X[t-1]` và `X[t]` để model tự tính tendency.
3. **Tách static vars ra khỏi output**: `lsm` và `geopotential_surf` chỉ là input feature (time-invariant), không còn là target.

**Bonus nhỏ**: `mesh_static` register buffer (không tạo tensor mỗi forward); NASA Tb có thêm mask channel (1 nếu valid, 0 nếu NaN) thay vì `nan_to_num(0)` âm thầm.

**Input layout ở mỗi sample**:
`[dyn_t, dyn_{t-1}, tb_t, tbmask_t, tb_{t-1}, tbmask_{t-1}, era5_static, pos_static]`
Quan trọng: **`dyn_t` đứng đầu** → `model.forward` dùng `x[..., :n_out]` làm residual base.

**Checkpoint format**: config lưu thêm `n_dyn_vars` để nb5 reconstruct đúng.


---
**⚡ Bản tối ưu Kaggle:** (1) tiền-giải-nén Zarr→memmap fp16 chạy 1 lần, (2) GNN processor vectorized theo batch, (3) AMP mixed-precision, (4) DataLoader đa luồng. Checkpoint giữ nguyên định dạng cho nb5.

In [1]:
!pip install torch zarr xarray dask numpy scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.2 MB/s eta 0:00:00


In [2]:
import os, json, math
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # giảm phân mảnh VRAM
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import xarray as xr

# ── Paths ─────────────────────────────────────────────────────────
STATS_JSON = "/kaggle/input/notebooks/phongngtun/stat-compute/era5_normalization_stats.json"
OUT      = "/kaggle/working"

# Thư mục SCRATCH để chứa memmap fp16 (KHÔNG tính vào 20GB output của Kaggle).
# /kaggle/tmp tồn tại trên Kaggle; fallback /tmp nếu chạy nơi khác.
SCRATCH = "/kaggle/tmp/cache_mm" if os.path.isdir("/kaggle") else "/tmp/cache_mm"
os.makedirs(SCRATCH, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True          # autotune conv/scatter kernels
torch.backends.cuda.matmul.allow_tf32 = True   # TF32 trên Ampere+ (vô hại trên T4/P100)
print(f"✓ Device: {DEVICE}")

# ── Cấu hình Năm ──────────────────────────────────────────────────
TRAIN_YEARS = [2020, 2021, 2022, 2023, 2024]
VAL_YEARS   = [2025]
TEST_YEARS  = [2026]

# ── Biến static (chỉ làm input) ───────────────────────────────────
STATIC_VARS  = ['lsm', 'geopotential_surf']

# ── Hyperparams ───────────────────────────────────────────────────
HIDDEN_DIM   = 128
N_LAYERS     = 4
BATCH_SIZE   = 4           # T4 16GB: g2m/m2g tạo tensor (B,128k cạnh,hidden) rất nặng → giữ 4 (thử 6-8 nếu còn VRAM)
LR           = 1e-4
N_EPOCHS     = 50
PATIENCE     = 7
MESH_RES     = 2.0

# ── Tối ưu I/O & tốc độ ───────────────────────────────────────────
USE_AMP      = True        # mixed precision (fp16) — nhanh hơn, ít VRAM hơn
NUM_WORKERS  = 4           # đọc memmap song song (memmap fork-safe trên Linux)
PREFETCH     = 4
MM_BLOCK     = 100         # đọc Zarr theo khối 100 bước = khớp chunk gốc {'time':100}

print(f"  Hidden dim : {HIDDEN_DIM} | GNN layers : {N_LAYERS}")
print(f"  Batch size : {BATCH_SIZE} | Epochs : {N_EPOCHS} | AMP : {USE_AMP}")
print(f"  Workers    : {NUM_WORKERS} | Scratch : {SCRATCH}")
print(f"  Static vars: {STATIC_VARS}")


✓ Device: cuda
  Hidden dim : 128 | GNN layers : 4
  Batch size : 4 | Epochs : 50 | AMP : True
  Workers    : 4 | Scratch : /kaggle/tmp/cache_mm
  Static vars: ['lsm', 'geopotential_surf']


## 4.1 Load Data & Build Graph

In [3]:
import os

print("--- Load multi-year data (lazy & auto-search Kaggle input) ---")

def find_zarr_path(folder_name):
    """
    Quét sâu (deep scan) toàn bộ thư mục /kaggle/input để tìm chính xác
    thư mục Zarr, bất chấp việc Kaggle có tạo thêm thư mục con hay không.
    """
    for root, dirs, files in os.walk('/kaggle/input'):
        if folder_name in dirs:
            return os.path.join(root, folder_name)
            
    raise FileNotFoundError(f"❌ Không tìm thấy: {folder_name}. Hãy kiểm tra xem bạn đã add đủ input chưa!")

def load_split(years):
    xs, labels, eyes = [], [], []
    for y in years:
        # Tự động quét đường dẫn thực tế trên đĩa Kaggle
        path_x     = find_zarr_path(f"brain_{y}_x.zarr")
        path_label = find_zarr_path(f"brain_{y}_label.zarr")
        path_eye   = find_zarr_path(f"eye_{y}_norm.zarr")
        
        # Mở Zarr lười biếng (lazy loading)
        xs.append(xr.open_zarr(path_x, consolidated=True))
        labels.append(xr.open_zarr(path_label, consolidated=True))
        eyes.append(xr.open_zarr(path_eye, consolidated=True))
    
    # Nối các năm lại với nhau theo trục thời gian
    ds_x = xr.concat(xs, dim='time')
    ds_label = xr.concat(labels, dim='time')
    ds_eye = xr.concat(eyes, dim='time')
    return ds_x, ds_label, ds_eye

# Khởi tạo Train và Val
ds_train_x, ds_train_label, ds_eye_train = load_split(TRAIN_YEARS)
ds_val_x, ds_val_label, ds_eye_val       = load_split(VAL_YEARS)

lats = ds_train_x.lat.values
lons = ds_train_x.lon.values
ALL_VARS = list(ds_train_x.data_vars)

# ── Tách dynamic vs static ──────────────────────────────────────
DYNAMIC_VARS = [v for v in ALL_VARS if v not in STATIC_VARS]
STATIC_VARS_PRESENT = [v for v in STATIC_VARS if v in ALL_VARS]

print(f"  All ERA5 vars  : {ALL_VARS}")
print(f"  Dynamic (Y)    : {DYNAMIC_VARS}")
print(f"  Static (X only): {STATIC_VARS_PRESENT}")
print(f"  Grid           : {len(lats)} lat × {len(lons)} lon = {len(lats)*len(lons)}")
print(f"  Train samples  : {len(ds_train_x.time)} (từ {len(TRAIN_YEARS)} năm)")
print(f"  Val samples    : {len(ds_val_x.time)} (từ {len(VAL_YEARS)} năm)")

--- Load multi-year data (lazy & auto-search Kaggle input) ---
  All ERA5 vars  : ['d2m', 'geopotential_surf', 'lsm', 'msl', 'q', 't2m', 'tp', 'u', 'u10', 'v', 'v10', 'z']
  Dynamic (Y)    : ['d2m', 'msl', 'q', 't2m', 'tp', 'u', 'u10', 'v', 'v10', 'z']
  Static (X only): ['lsm', 'geopotential_surf']
  Grid           : 159 lat × 201 lon = 31959
  Train samples  : 7303 (từ 5 năm)
  Val samples    : 1459 (từ 1 năm)


In [4]:
# ── Build regular mesh (2°) ───────────────────────────────────────
print("--- Build mesh ---")
from scipy.spatial import cKDTree

mesh_lats_1d = np.arange(-10.0, 28.5, MESH_RES)
mesh_lons_1d = np.arange(92.0,  142.0, MESH_RES)
mesh_lat_grid, mesh_lon_grid = np.meshgrid(mesh_lats_1d, mesh_lons_1d, indexing='ij')
mesh_lats_flat = mesh_lat_grid.flatten()
mesh_lons_flat = mesh_lon_grid.flatten()
N_MESH = len(mesh_lats_flat)
print(f"  Mesh nodes : {len(mesh_lats_1d)} × {len(mesh_lons_1d)} = {N_MESH}")

# Static features for mesh nodes: sin/cos lat/lon
lat_r = np.deg2rad(mesh_lats_flat)
lon_r = np.deg2rad(mesh_lons_flat)
mesh_static_np = np.stack([
    np.sin(lat_r), np.cos(lat_r),
    np.sin(lon_r), np.cos(lon_r),
], axis=1).astype(np.float32)   # (N_MESH, 4)

# ── Grid → Mesh edges (KNN k=4) ───────────────────────────────────
grid_lat_g, grid_lon_g = np.meshgrid(lats, lons, indexing='ij')
grid_pts = np.stack([grid_lat_g.flatten(), grid_lon_g.flatten()], axis=1)
mesh_pts = np.stack([mesh_lats_flat, mesh_lons_flat], axis=1)

tree = cKDTree(mesh_pts)
dists, indices = tree.query(grid_pts, k=4)
n_grid = len(grid_pts)

src_g2m = np.repeat(np.arange(n_grid), 4)
dst_g2m = indices.flatten()
dist_g2m = dists.flatten()
dist_g2m_norm = (dist_g2m - dist_g2m.mean()) / (dist_g2m.std() + 1e-6)

g2m_src = torch.tensor(src_g2m, dtype=torch.long)
g2m_dst = torch.tensor(dst_g2m, dtype=torch.long)
g2m_edge_attr = torch.tensor(dist_g2m_norm, dtype=torch.float32).unsqueeze(1)

# ── Mesh → Mesh edges (8-connectivity) ───────────────────────────
n_lat_m = len(mesh_lats_1d)
n_lon_m = len(mesh_lons_1d)
m2m_src_list, m2m_dst_list = [], []
for i in range(n_lat_m):
    for j in range(n_lon_m):
        node = i * n_lon_m + j
        for di in [-1, 0, 1]:
            for dj in [-1, 0, 1]:
                if di == 0 and dj == 0: continue
                ni, nj = i + di, j + dj
                if 0 <= ni < n_lat_m and 0 <= nj < n_lon_m:
                    m2m_src_list.append(node)
                    m2m_dst_list.append(ni * n_lon_m + nj)
m2m_src = torch.tensor(m2m_src_list, dtype=torch.long)
m2m_dst = torch.tensor(m2m_dst_list, dtype=torch.long)

# ── Mesh → Grid edges (reverse of g2m) ────────────────────────────
m2g_src = g2m_dst.clone()
m2g_dst = g2m_src.clone()
m2g_edge_attr = g2m_edge_attr.clone()

print(f"  G→M edges  : {len(g2m_src):,}")
print(f"  M→M edges  : {len(m2m_src):,}")
print(f"  M→G edges  : {len(m2g_src):,}")
print("✓ Graph built")


--- Build mesh ---
  Mesh nodes : 20 × 25 = 500
  G→M edges  : 127,836
  M→M edges  : 3,734
  M→G edges  : 127,836
✓ Graph built


In [5]:
# ════════════════════════════════════════════════════════════════════
#  TIỀN-GIẢI-NÉN Zarr → memmap fp16 (CHẠY 1 LẦN cho mỗi split)
#  ─ Lý do: Zarr được lưu với chunk {'time':100,...} nên đọc lẻ 1 timestep
#    phải giải nén cả khối 100 bước → thừa ~100×, lặp lại mỗi epoch.
#  ─ Cách: quét tuần tự theo khối (mỗi chunk giải nén đúng 1 lần), ghi fp16
#    ra ổ đĩa. Sau đó Dataset chỉ index → không giải nén lại, không OOM.
# ════════════════════════════════════════════════════════════════════
print("--- Materialize splits → fp16 memmap ---")

def _expand_channels(var_list, ds):
    ch = []
    for v in var_list:
        if v not in ds.data_vars: continue
        da = ds[v]
        if 'pressure_level' in da.dims:
            for lev in da.pressure_level.values:
                ch.append((v, int(lev)))
        else:
            ch.append((v, None))
    return ch

def _read_block(ds, channels, t0, t1):
    """Đọc [t0:t1) cho mọi kênh → (blk, n_grid, C) fp32 (đã nan_to_num)."""
    blk = t1 - t0
    arrs = []
    for v, lev in channels:
        da = ds[v].isel(time=slice(t0, t1))
        if lev is not None:
            da = da.sel(pressure_level=lev)
        a = da.values.astype(np.float32).reshape(blk, -1)   # decompress 1 lần
        arrs.append(a)
    out = np.stack(arrs, axis=-1)                            # (blk, n_grid, C)
    return np.nan_to_num(out, copy=False)

def materialize_split(ds_x, ds_label, ds_eye, dyn_vars, lats, lons, tag):
    n_grid = len(lats) * len(lons)
    dyn_ch = _expand_channels(dyn_vars, ds_x)
    n_dyn  = len(dyn_ch)
    T      = len(ds_x.time)

    paths = {
        'dyn':   f"{SCRATCH}/{tag}_dyn_{T}x{n_grid}x{n_dyn}.f16",
        'label': f"{SCRATCH}/{tag}_label_{T}x{n_grid}x{n_dyn}.f16",
        'tb':    f"{SCRATCH}/{tag}_tb_{T}x{n_grid}.f16",
        'mask':  f"{SCRATCH}/{tag}_mask_{T}x{n_grid}.f16",
    }
    done_flag = f"{SCRATCH}/{tag}_DONE_{T}_{n_grid}_{n_dyn}"

    if os.path.exists(done_flag):
        print(f"  [{tag}] đã có cache (T={T}, C={n_dyn}) → bỏ qua decode.")
    else:
        dyn_mm   = np.memmap(paths['dyn'],   dtype=np.float16, mode='w+', shape=(T, n_grid, n_dyn))
        label_mm = np.memmap(paths['label'], dtype=np.float16, mode='w+', shape=(T, n_grid, n_dyn))
        tb_mm    = np.memmap(paths['tb'],    dtype=np.float16, mode='w+', shape=(T, n_grid))
        mask_mm  = np.memmap(paths['mask'],  dtype=np.float16, mode='w+', shape=(T, n_grid))

        for t0 in range(0, T, MM_BLOCK):
            t1 = min(t0 + MM_BLOCK, T)
            dyn_mm[t0:t1]   = _read_block(ds_x,     dyn_ch, t0, t1).astype(np.float16)
            label_mm[t0:t1] = _read_block(ds_label, dyn_ch, t0, t1).astype(np.float16)
            tb_raw = ds_eye['Tb'].isel(time=slice(t0, t1)).values.astype(np.float32).reshape(t1 - t0, -1)
            mask_mm[t0:t1] = (~np.isnan(tb_raw)).astype(np.float16)
            tb_mm[t0:t1]   = np.nan_to_num(tb_raw, nan=0.0, copy=False).astype(np.float16)
            print(f"    [{tag}] {t1}/{T}", end='\r', flush=True)

        dyn_mm.flush(); label_mm.flush(); tb_mm.flush(); mask_mm.flush()
        del dyn_mm, label_mm, tb_mm, mask_mm
        open(done_flag, 'w').close()
        gb = (T * n_grid * (n_dyn * 2 + 1)) * 2 / 1e9
        print(f"\n  [{tag}] xong. ~{gb:.1f} GB fp16 trên đĩa, {n_dyn} kênh động lực.")

    # ── static (lsm/geopot + sin/cos lat-lon) — nhỏ, giữ trong RAM ──
    static_ch = _expand_channels(STATIC_VARS_PRESENT, ds_x)
    if static_ch:
        ds0 = ds_x.isel(time=0)
        arrs = []
        for v, lev in static_ch:
            da = ds0[v]
            if lev is not None: da = da.sel(pressure_level=lev)
            arrs.append(da.values.flatten())
        stat_e5 = np.stack(arrs, axis=1).astype(np.float32)
    else:
        stat_e5 = np.zeros((n_grid, 0), dtype=np.float32)
    lat_r, lon_r = np.deg2rad(lats), np.deg2rad(lons)
    lat_g, lon_g = np.meshgrid(lat_r, lon_r, indexing='ij')
    pos = np.stack([np.sin(lat_g), np.cos(lat_g),
                    np.sin(lon_g), np.cos(lon_g)], axis=-1).reshape(n_grid, 4).astype(np.float32)
    static = np.nan_to_num(np.concatenate([stat_e5, pos], axis=1), copy=False)

    return {'paths': paths, 'T': T, 'n_grid': n_grid, 'n_dyn': n_dyn,
            'static': torch.from_numpy(static)}

train_mm = materialize_split(ds_train_x, ds_train_label, ds_eye_train, DYNAMIC_VARS, lats, lons, 'train')
val_mm   = materialize_split(ds_val_x,   ds_val_label,   ds_eye_val,   DYNAMIC_VARS, lats, lons, 'val')
print("✓ Materialize hoàn tất.")


--- Materialize splits → fp16 memmap ---
    [train] 7303/7303
  [train] xong. ~13.5 GB fp16 trên đĩa, 14 kênh động lực.
    [val] 1459/1459
  [val] xong. ~2.7 GB fp16 trên đĩa, 14 kênh động lực.
✓ Materialize hoàn tất.


## 4.2 Dataset (2-timestep input, static-as-feature, NaN mask)

In [6]:
class WeatherDatasetMM(Dataset):
    """Đọc từ memmap fp16 đã giải nén sẵn → __getitem__ chỉ là indexing.
       Không giải nén Zarr, không OOM (memmap backed-by-disk, OS tự phân trang)."""
    def __init__(self, mm):
        self.T       = mm['T']
        self.n_grid  = mm['n_grid']
        self.n_dyn   = mm['n_dyn']
        self.paths   = mm['paths']
        self.static  = mm['static']                 # (n_grid, n_static) fp32
        self.n_static_total = self.static.shape[1]
        self.n_in  = self.n_dyn * 2 + 4 + self.n_static_total
        self.n_out = self.n_dyn
        self._mm = None                              # mở lười trong từng worker
        print(f"    n_in={self.n_in} (2×{self.n_dyn} dyn + 4 nasa + {self.n_static_total} static) | n_out={self.n_out}")

    def _ensure_open(self):
        if self._mm is None:
            T, n_grid, n_dyn = self.T, self.n_grid, self.n_dyn
            self._mm = {
                'dyn':   np.memmap(self.paths['dyn'],   dtype=np.float16, mode='r', shape=(T, n_grid, n_dyn)),
                'label': np.memmap(self.paths['label'], dtype=np.float16, mode='r', shape=(T, n_grid, n_dyn)),
                'tb':    np.memmap(self.paths['tb'],    dtype=np.float16, mode='r', shape=(T, n_grid)),
                'mask':  np.memmap(self.paths['mask'],  dtype=np.float16, mode='r', shape=(T, n_grid)),
            }

    def __len__(self):
        return self.T - 1

    def __getitem__(self, idx):
        self._ensure_open()
        t = idx + 1
        mm = self._mm
        dyn_t   = torch.from_numpy(np.asarray(mm['dyn'][t],     dtype=np.float32))
        dyn_tm1 = torch.from_numpy(np.asarray(mm['dyn'][t - 1], dtype=np.float32))
        y       = torch.from_numpy(np.asarray(mm['label'][t],   dtype=np.float32))
        tb_t    = torch.from_numpy(np.asarray(mm['tb'][t],      dtype=np.float32)).unsqueeze(1)
        tb_tm1  = torch.from_numpy(np.asarray(mm['tb'][t - 1],  dtype=np.float32)).unsqueeze(1)
        mk_t    = torch.from_numpy(np.asarray(mm['mask'][t],    dtype=np.float32)).unsqueeze(1)
        mk_tm1  = torch.from_numpy(np.asarray(mm['mask'][t - 1],dtype=np.float32)).unsqueeze(1)

        x = torch.cat([dyn_t, dyn_tm1, tb_t, mk_t, tb_tm1, mk_tm1, self.static], dim=-1)
        return x, y


train_dataset = WeatherDatasetMM(train_mm)
val_dataset   = WeatherDatasetMM(val_mm)

assert train_dataset.n_in  == val_dataset.n_in
assert train_dataset.n_out == val_dataset.n_out
N_IN, N_OUT, N_DYN = train_dataset.n_in, train_dataset.n_out, train_dataset.n_dyn

_loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=PREFETCH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  **_loader_kw)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, **_loader_kw)

print(f"\n✓ Train: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"  Val  : {len(val_dataset)} samples, {len(val_loader)} batches")


    n_in=38 (2×14 dyn + 4 nasa + 6 static) | n_out=14
    n_in=38 (2×14 dyn + 4 nasa + 6 static) | n_out=14

✓ Train: 7302 samples, 1826 batches
  Val  : 1458 samples, 365 batches


## 4.3 Model (residual output + mesh_static buffer)

In [7]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=None, layers=2):
        super().__init__()
        hidden = hidden or out_dim
        net = [nn.Linear(in_dim, hidden), nn.SiLU()]
        for _ in range(layers - 2):
            net += [nn.Linear(hidden, hidden), nn.SiLU()]
        net += [nn.Linear(hidden, out_dim)]
        self.net = nn.Sequential(*net)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        return self.norm(self.net(x))


class GNNLayer(nn.Module):
    """★ VECTORIZED: xử lý cả batch (B,N,D) cùng lúc — bỏ vòng lặp for b in range(B)."""
    def __init__(self, node_dim, edge_dim=1):
        super().__init__()
        self.msg_mlp  = MLP(node_dim * 2 + edge_dim, node_dim)
        self.node_mlp = MLP(node_dim * 2, node_dim)

    def forward(self, x, edge_src, edge_dst, edge_attr, deg):
        # x: (B, N, D) ; edge_*: (E,) ; edge_attr: (E, edim) ; deg: (1, N, 1)
        B, N, D = x.shape
        src = x[:, edge_src, :]                                  # (B, E, D)
        dst = x[:, edge_dst, :]                                  # (B, E, D)
        ea  = edge_attr.unsqueeze(0).expand(B, -1, -1)           # (B, E, edim)
        msg = self.msg_mlp(torch.cat([src, dst, ea], dim=-1))    # (B, E, D)

        agg = torch.zeros(B, N, D, device=x.device, dtype=x.dtype)
        idx = edge_dst.view(1, -1, 1).expand(B, -1, D)
        agg.scatter_add_(1, idx, msg.to(x.dtype))
        agg = agg / (deg.to(x.dtype) + 1e-6)

        return x + self.node_mlp(torch.cat([x, agg], dim=-1))


class GraphCastSEA(nn.Module):
    def __init__(self, n_in, n_out, n_mesh, mesh_static_np,
                 m2m_src, m2m_dst, hidden=128, n_layers=4):
        super().__init__()
        self.n_mesh = n_mesh
        self.n_out  = n_out

        self.register_buffer('mesh_static', torch.from_numpy(mesh_static_np).float())

        # bậc vào (in-degree) của m2m — hằng số, tính 1 lần
        deg = torch.zeros(n_mesh, 1)
        deg.scatter_add_(0, m2m_dst.view(-1, 1), torch.ones(m2m_dst.numel(), 1))
        self.register_buffer('m2m_deg', deg.view(1, n_mesh, 1))
        self.register_buffer('m2m_ea_zero', torch.zeros(m2m_src.numel(), 1))

        self.grid_encoder = MLP(n_in, hidden)
        self.mesh_encoder = MLP(4,    hidden)
        self.g2m_mlp = MLP(hidden * 2 + 1, hidden)
        self.processors = nn.ModuleList([GNNLayer(hidden, edge_dim=1) for _ in range(n_layers)])
        self.m2g_mlp = MLP(hidden * 2 + 1, hidden)
        self.decoder = nn.Linear(hidden, n_out)
        nn.init.zeros_(self.decoder.weight)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, x_grid,
                g2m_src, g2m_dst, g2m_ea,
                m2m_src, m2m_dst,
                m2g_src, m2g_dst, m2g_ea):
        B = x_grid.size(0)
        device = x_grid.device

        residual_base = x_grid[..., :self.n_out]
        h_grid = self.grid_encoder(x_grid)
        h_mesh = self.mesh_encoder(self.mesh_static).unsqueeze(0).expand(B, -1, -1).contiguous()

        # G→M
        g_feat = h_grid[:, g2m_src, :]
        m_feat = h_mesh[:, g2m_dst, :]
        ea     = g2m_ea.unsqueeze(0).expand(B, -1, -1)
        msg_g2m = self.g2m_mlp(torch.cat([g_feat, m_feat, ea], dim=-1))
        agg = torch.zeros(B, self.n_mesh, msg_g2m.size(-1), device=device, dtype=h_mesh.dtype)
        idx = g2m_dst.view(1, -1, 1).expand(B, -1, msg_g2m.size(-1))
        agg.scatter_add_(1, idx, msg_g2m.to(h_mesh.dtype))
        h_mesh = h_mesh + agg

        # Processor — batched, không còn vòng lặp Python
        for proc in self.processors:
            h_mesh = proc(h_mesh, m2m_src, m2m_dst, self.m2m_ea_zero, self.m2m_deg)

        # M→G
        n_grid_ = h_grid.size(1)
        m_feat2 = h_mesh[:, m2g_src, :]
        g_feat2 = h_grid[:, m2g_dst, :]
        ea2     = m2g_ea.unsqueeze(0).expand(B, -1, -1)
        msg_m2g = self.m2g_mlp(torch.cat([m_feat2, g_feat2, ea2], dim=-1))
        agg2 = torch.zeros(B, n_grid_, msg_m2g.size(-1), device=device, dtype=h_grid.dtype)
        idx2 = m2g_dst.view(1, -1, 1).expand(B, -1, msg_m2g.size(-1))
        agg2.scatter_add_(1, idx2, msg_m2g.to(h_grid.dtype))
        h_grid = h_grid + agg2

        delta = self.decoder(h_grid)
        return residual_base + delta


model = GraphCastSEA(
    n_in=N_IN, n_out=N_OUT, n_mesh=N_MESH,
    mesh_static_np=mesh_static_np,
    m2m_src=m2m_src, m2m_dst=m2m_dst,
    hidden=HIDDEN_DIM, n_layers=N_LAYERS,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model: {n_params:,} parameters")
print(f"  N_IN={N_IN}, N_OUT={N_OUT}, n_mesh={N_MESH}")
print(f"  Residual: YES | Past timesteps: 2 | Static in input: {STATIC_VARS_PRESENT}")


✓ Model: 538,382 parameters
  N_IN=38, N_OUT=14, n_mesh=500
  Residual: YES | Past timesteps: 2 | Static in input: ['lsm', 'geopotential_surf']


## 4.4 Training

In [8]:
# ── Latitude-weighted RMSE loss ───────────────────────────────────
lat_weights = torch.tensor(np.cos(np.deg2rad(lats)), dtype=torch.float32)
lat_weights = lat_weights / lat_weights.mean()
lat_w_grid = lat_weights.unsqueeze(1).expand(-1, len(lons)).reshape(-1).to(DEVICE)

def weighted_rmse(pred, target):
    se = (pred.float() - target.float()) ** 2          # loss tính ở fp32 cho ổn định
    w  = lat_w_grid.unsqueeze(0).unsqueeze(-1)
    return torch.sqrt((se * w).mean())

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

# ── Graph tensors → device ────────────────────────────────────────
g2m_src_d = g2m_src.to(DEVICE); g2m_dst_d = g2m_dst.to(DEVICE); g2m_ea_d = g2m_edge_attr.to(DEVICE)
m2m_src_d = m2m_src.to(DEVICE); m2m_dst_d = m2m_dst.to(DEVICE)
m2g_src_d = m2g_src.to(DEVICE); m2g_dst_d = m2g_dst.to(DEVICE); m2g_ea_d = m2g_edge_attr.to(DEVICE)

def run_model(x):
    return model(x, g2m_src_d, g2m_dst_d, g2m_ea_d,
                 m2m_src_d, m2m_dst_d,
                 m2g_src_d, m2g_dst_d, m2g_ea_d)

# ── Persistence baseline ──────────────────────────────────────────
print("--- Persistence baseline (val) ---")
with torch.no_grad():
    losses = []
    for x_batch, y_batch in val_loader:
        x_batch = x_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)
        losses.append(weighted_rmse(x_batch[..., :N_OUT], y_batch).item())
    persist_val = float(np.mean(losses))
print(f"  Persistence val loss: {persist_val:.4f}")

# ── Training loop ─────────────────────────────────────────────────
print("\n--- Bắt đầu training ---")
history = {'train_loss': [], 'val_loss': [], 'lr': [], 'persistence_val': persist_val}
best_val = float('inf'); patience_cnt = 0

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_losses = []
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            pred = run_model(x_batch)
            loss = weighted_rmse(pred, y_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                pred = run_model(x_batch)
                vl = weighted_rmse(pred, y_batch)
            val_losses.append(vl.item())

    train_loss = float(np.mean(train_losses)); val_loss = float(np.mean(val_losses))
    lr_now = scheduler.get_last_lr()[0]; scheduler.step()
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss); history['lr'].append(lr_now)

    beats = "✓" if val_loss < persist_val else "✗"
    print(f"Epoch {epoch:3d}/{N_EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f} "
          f"(persist={persist_val:.4f} {beats}) | lr={lr_now:.2e}")

    if val_loss < best_val:
        best_val = val_loss; patience_cnt = 0
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': best_val, 'persistence_val': persist_val,
            'config': {'n_in': N_IN, 'n_out': N_OUT, 'n_dyn': N_DYN,
                       'n_mesh': N_MESH, 'hidden': HIDDEN_DIM, 'n_layers': N_LAYERS,
                       'dynamic_vars': DYNAMIC_VARS, 'static_vars': STATIC_VARS_PRESENT,
                       'mesh_res': MESH_RES, 'n_past': 2, 'residual': True},
            'mesh_static_np': mesh_static_np,
        }, f'{OUT}/graphcast_sea_best.pt')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}"); break

with open(f'{OUT}/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f"\n✅ Training xong! Best val={best_val:.4f} | Persistence={persist_val:.4f} "
      f"| Improvement={(1-best_val/persist_val)*100:+.1f}%")
print(f"   Checkpoint: {OUT}/graphcast_sea_best.pt")


--- Persistence baseline (val) ---
  Persistence val loss: 0.5461

--- Bắt đầu training ---
Epoch   1/50 | train=0.4244 | val=0.4142 (persist=0.5461 ✓) | lr=1.00e-04
Epoch   2/50 | train=0.3934 | val=0.4006 (persist=0.5461 ✓) | lr=9.99e-05
Epoch   3/50 | train=0.3845 | val=0.3931 (persist=0.5461 ✓) | lr=9.96e-05
Epoch   4/50 | train=0.3787 | val=0.3903 (persist=0.5461 ✓) | lr=9.91e-05
Epoch   5/50 | train=0.3749 | val=0.3858 (persist=0.5461 ✓) | lr=9.84e-05
Epoch   6/50 | train=0.3721 | val=0.3841 (persist=0.5461 ✓) | lr=9.76e-05
Epoch   7/50 | train=0.3698 | val=0.3806 (persist=0.5461 ✓) | lr=9.65e-05
Epoch   8/50 | train=0.3679 | val=0.3832 (persist=0.5461 ✓) | lr=9.52e-05
Epoch   9/50 | train=0.3663 | val=0.3772 (persist=0.5461 ✓) | lr=9.38e-05
Epoch  10/50 | train=0.3648 | val=0.3762 (persist=0.5461 ✓) | lr=9.22e-05
Epoch  11/50 | train=0.3633 | val=0.3740 (persist=0.5461 ✓) | lr=9.05e-05
Epoch  12/50 | train=0.3620 | val=0.3726 (persist=0.5461 ✓) | lr=8.85e-05
Epoch  13/50 | train